In [1]:
import os
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import re
import openpyxl
import numpy as np
np.set_printoptions(legacy="1.13")

In [ ]:
folder=r'Path'
OFolder=r'Output Folder Path'

In [ ]:
file_list=[]
for (root, dirs, file) in os.walk(folder):
    for f in file:
        if ('.xlsx') in f:

                    file_list.append(f)
file_list

In [4]:
file_link=[]

for i in range(len(file_list)):
     for r,d,f in os.walk(folder):
          for files in f:
               if files == file_list[i]:
                    file_link.append(os.path.join(r,files))

In [5]:
len(file_link)

10

In [ ]:
file_link[1]

In [7]:
cols=['Brand']
df_s = pd.DataFrame(columns=cols)

In [8]:
file_link[1].split("_PIES")[0].split("Excel\\")[1]

'BULLDOG'

In [ ]:
for i in range(len(file_link)):
    # workbook=openpyxl.load_workbook(file_link[i])
    BrandName= file_link[i].split("_PIES")[0].split("Excel\\")[1]
    print(i,BrandName)
    df_item=pd.read_excel(file_link[i],sheet_name="Items")
    df_Attr=pd.read_excel(file_link[i],sheet_name="ProductAttributes")
    df_item=df_item.astype(str)
    df_Attr=df_Attr.astype(str)
    df=df_Attr.merge(df_item,how="inner")
    ListColumns=['PartNumber','Description_SHO', 'AttributeID',"Value"]
    df=df[ListColumns]
    df=df.drop_duplicates().reset_index(drop=True)
    df['Brand']=BrandName
    df_s=pd.concat([df_s,df])   


In [ ]:
df_s

In [ ]:
df_autocare=pd.read_csv(r"path/file.csv")

In [ ]:
df_autocare

In [ ]:
df_autocare=df_autocare[['PAID',"PAName"]].drop_duplicates().reset_index(drop=True)
df_autocare
df_autocare["PAID"]=df_autocare["PAID"].astype(str)
for i in range(len(df_autocare)):
    df_autocare.loc[i,"PAID"]=df_autocare["PAID"][i].replace('.0', '')
df_autocare.head()

In [ ]:
df_s.info()

In [15]:
df_Match=df_s.merge(df_autocare,left_on="AttributeID",right_on="PAID",how='left')

In [ ]:
df_Match

In [17]:
df_Match['PAName']=np.where(df_Match['PAName'].isnull(), df_Match['AttributeID'], df_Match['PAName'])

In [18]:
df_Match.columns

Index(['Brand', 'PartNumber', 'Description_SHO', 'AttributeID', 'Value',
       'PAID', 'PAName'],
      dtype='object')

In [19]:
df_Match.rename(columns={"Description_SHO":"PartTerminologyName"},inplace=True)
df_Match=df_Match[['Brand', 'PartNumber', 'PartTerminologyName',  'PAName', 'Value']]

In [20]:
chunk_size=1000000
# Create a list of DataFrames by splitting the original DataFrame
df_chunks = [df_Match.iloc[i:i + chunk_size] for i in range(0, len(df_Match), chunk_size)]

In [21]:
df_Attributes=df_Match[['Brand', 'PartTerminologyName',  'PAName']].drop_duplicates().reset_index(drop=True)

In [22]:
with pd.ExcelWriter(OFolder+'\\'+'Horizon_Global_Attribute_List.xlsx') as writer:  # doctest: +SKIP
    for i, chunk in enumerate(df_chunks):
        sheet_name = f"Chunk_{i+1}"  # Naming each sheet dynamically
        chunk.to_excel(writer, sheet_name=sheet_name, index=False)
    df_Attributes.to_excel(writer,index=False, sheet_name='FBG_Attributes')